# EX — Deep Learning Advanced Real-World Exercises

Multi-head attention, dropout/batchnorm, an autoencoder for anomaly detection, a GAN
sketch, and Q-learning. Requires `torch`.


## 1. Multi-Head Self-Attention (the core Transformer block)
**Pointer:** each head learns to attend to different kinds of relationships; concatenating heads gives the model multiple 'perspectives' at once.

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(0)

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, d_model*3)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.d_head).permute(2,0,3,1,4)
        q, k, v = qkv[0], qkv[1], qkv[2]   # each (B, n_heads, T, d_head)
        scores = (q @ k.transpose(-2,-1)) / (self.d_head ** 0.5)
        attn = torch.softmax(scores, dim=-1)
        out = attn @ v                       # (B, n_heads, T, d_head)
        out = out.transpose(1,2).reshape(B, T, D)
        return self.out(out)

mha = MultiHeadSelfAttention(d_model=16, n_heads=4)
x = torch.rand(2, 5, 16)  # batch=2, seq_len=5, d_model=16
print(mha(x).shape)  # expect (2, 5, 16)


### TODO 1
Wrap this attention layer into a full Transformer block: `x = x + attention(x)` (residual), then `LayerNorm`, then a feedforward `Linear -> ReLU -> Linear`, then another residual + LayerNorm.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        # TODO: define self.ff (Linear->ReLU->Linear) and self.norm2
    def forward(self, x):
        # TODO: residual + norm around attention, then residual + norm around feedforward
        pass

block = TransformerBlock(d_model=16, n_heads=4, d_ff=32)


<details><summary>Solution</summary>

```python
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm1(x + self.attn(x))
        x = self.norm2(x + self.ff(x))
        return x

block = TransformerBlock(16, 4, 32)
print(block(x).shape)
```
</details>


## 2. Regularization — Dropout & Batch Norm

In [ ]:
model = nn.Sequential(
    nn.Linear(20, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, 2),
)

sample = torch.rand(8, 20)
model.train()
out_train = model(sample)   # dropout active, batchnorm uses batch statistics
model.eval()
out_eval = model(sample)    # dropout off, batchnorm uses running statistics
print("train/eval outputs differ:", not torch.allclose(out_train, out_eval))


### TODO 2
Explain in a comment: why would validation loss look *artificially better* if you forgot to call `model.eval()` before computing it? (Think about what dropout does during `.train()` mode.)

In [ ]:
# TODO: your explanation as a comment


<details><summary>Discussion</summary>

Dropout randomly zeroes activations during training mode, which is a form of noise/regularization — it doesn't reflect the model's true, full-capacity prediction. Leaving `.train()` mode on during validation makes the loss noisier and not directly comparable to true inference-time performance.
</details>

## 3. Autoencoder for Anomaly Detection
Real-world use: fraud/defect detection — train only on 'normal' data, flag high reconstruction error as anomalous.

In [ ]:
import numpy as np
np.random.seed(0)

# Normal data: clustered around specific patterns
normal_data = np.random.normal(0, 1, (500, 10)).astype(np.float32)
# Anomalies: shifted / different distribution
anomalies = (np.random.normal(0, 1, (20, 10)) + 6).astype(np.float32)

class Autoencoder(nn.Module):
    def __init__(self, in_dim, bottleneck=3):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(in_dim, 8), nn.ReLU(), nn.Linear(8, bottleneck))
        self.decoder = nn.Sequential(nn.Linear(bottleneck, 8), nn.ReLU(), nn.Linear(8, in_dim))
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

ae = Autoencoder(in_dim=10)
opt = torch.optim.Adam(ae.parameters(), lr=1e-2)
X = torch.tensor(normal_data)

for epoch in range(100):
    opt.zero_grad()
    recon = ae(X)
    loss = nn.functional.mse_loss(recon, X)
    loss.backward()
    opt.step()
print("final training reconstruction loss:", loss.item())


### TODO 3
Compute the per-sample reconstruction error (MSE) for both `normal_data` and `anomalies`, and confirm anomalies have noticeably higher error on average — this is the anomaly detection signal.

In [ ]:
# TODO
ae.eval()
with torch.no_grad():
    normal_errors = None
    anomaly_errors = None
print(normal_errors.mean().item() if normal_errors is not None else None,
      anomaly_errors.mean().item() if anomaly_errors is not None else None)


<details><summary>Solution</summary>

```python
ae.eval()
with torch.no_grad():
    normal_recon = ae(torch.tensor(normal_data))
    anomaly_recon = ae(torch.tensor(anomalies))
    normal_errors = ((normal_recon - torch.tensor(normal_data))**2).mean(dim=1)
    anomaly_errors = ((anomaly_recon - torch.tensor(anomalies))**2).mean(dim=1)
print(normal_errors.mean().item(), anomaly_errors.mean().item())
# anomaly_errors.mean() should be noticeably higher
```
</details>


## 4. GAN — Conceptual Sketch (Generator vs. Discriminator)
We won't fully train a GAN here (unstable/slow), but we'll wire up the two networks and one training step to see the adversarial structure.

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(noise_dim, 16), nn.ReLU(), nn.Linear(16, out_dim))
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x)

gen = Generator(noise_dim=4, out_dim=2)
disc = Discriminator(in_dim=2)

real_data = torch.tensor(np.random.normal(3, 0.5, (32, 2)).astype(np.float32))
noise = torch.rand(32, 4)
fake_data = gen(noise)

# One discriminator training step: real should score close to 1, fake close to 0
d_loss = nn.functional.binary_cross_entropy(disc(real_data), torch.ones(32,1)) + \
         nn.functional.binary_cross_entropy(disc(fake_data.detach()), torch.zeros(32,1))
print("discriminator loss:", d_loss.item())

# One generator training step: wants discriminator to score fake data close to 1 (fool it)
g_loss = nn.functional.binary_cross_entropy(disc(fake_data), torch.ones(32,1))
print("generator loss:", g_loss.item())


## Key Takeaways
- Multi-head attention lets a model attend to multiple relationship types simultaneously — the core Transformer building block.
- A full Transformer block wraps attention and a feedforward layer each in a residual connection + LayerNorm.
- Dropout is only active in `.train()` mode; always switch to `.eval()` before computing validation/test metrics.
- An autoencoder trained only on normal data naturally flags anomalies via high reconstruction error.
- A GAN trains two networks adversarially: the discriminator tries to catch fakes, the generator tries to fool it.
